In [156]:
import torch

data = torch.tensor(
    [
        [2, 3, 3],
        [5, 6, 4],
        [7, 8, 5]
    ]
)
data

tensor([[2, 3, 3],
        [5, 6, 4],
        [7, 8, 5]])

In [157]:
x = data[:,0:2]
y = data[:,2]

print(x)
print(y)

tensor([[2, 3],
        [5, 6],
        [7, 8]])
tensor([3, 4, 5])


In [158]:
# Check CUDA status
import torch
print('Torch:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
Torch CUDA build: 12.8
CUDA available: True
Device: NVIDIA GeForce RTX 3060 Laptop GPU


In [159]:
# Move tensor to GPU if available, else keep on CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
x = x.to(device)
y = y.to(device)
print('Using device:', device)
print('data device:', data.device)

Using device: cuda
data device: cuda:0


### Weight Initialization
Weight initialization is a crucial step in training neural networks, as it can significantly impact the convergence and performance of the model. Proper initialization can help prevent issues such as vanishing or exploding gradients, which can hinder the training process.
There are several common methods for weight initialization, including:
1. **Random Initialization**: Weights are initialized randomly, often using a uniform or normal distribution. This method can work but may lead to issues with convergence if not done carefully.
2. **Xavier Initialization**: Also known as Glorot initialization, this method initializes weights based on the number of input and output neurons. It helps maintain the variance of activations across layers, which can improve convergence.
3. **He Initialization**: This method is similar to Xavier initialization but is designed for layers with ReLU activation functions. It initializes weights based on the number of input neurons, which can help prevent vanishing gradients.
4. **Orthogonal Initialization**: Weights are initialized to be orthogonal, which can help maintain the stability of the network during training.
Choosing the right weight initialization method can depend on the architecture of the neural network and the activation functions used. It is often beneficial to experiment with different initialization methods to find the one that works best for a specific problem.

In [160]:
def initialize_parameters():
    parameters = {}
    parameters['w1']= torch.ones((2,2))*0.1
    parameters['b2']= torch.zeros((2,1))
    parameters['w2']= torch.ones((2,1))*0.1
    parameters['b2']= torch.zeros((1,1))
    return parameters

initialize_parameters()

{'w1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b2': tensor([[0.]]),
 'w2': tensor([[0.1000],
         [0.1000]])}

In [161]:
# & Better way to initialize parameters
def initialize_parameters(layer_dims):
    torch.manual_seed(3)  # For reproducibility
    parameters = {}
    L = len(layer_dims)  # Number of layers in the network
    for l in range(1, L):
        parameters['w' + str(l)] = torch.ones((layer_dims[l - 1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = torch.zeros((layer_dims[l], 1))

    return parameters    

initialize_parameters([2, 2, 1])

{'w1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'w2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

In [162]:

#* Forward Propagation
def linear_forward(A_prev, W, b):
    Z = torch.mm(W.T, A_prev) + b
    return Z

In [163]:
def L_layer_forward(X,parameters):
    A = X
    L = len(parameters) // 2  # Number of layers in the network
    for l in range(1,L+1):
        A_prev = A
        W = parameters['w' + str(l)]
        b = parameters['b' + str(l)]
        A = linear_forward(A_prev, W, b)
    return A,A_prev

In [164]:
params = initialize_parameters([2, 2, 1])
print(params)
params = {k: v.to(device) for k, v in params.items()}
print(params)
X_sample = x[0].reshape(2,1).float()
print(X_sample)
y_hat, A1 = L_layer_forward(X_sample, params)



{'w1': tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]]), 'b1': tensor([[0.],
        [0.]]), 'w2': tensor([[0.1000],
        [0.1000]]), 'b2': tensor([[0.]])}
{'w1': tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]], device='cuda:0'), 'b1': tensor([[0.],
        [0.]], device='cuda:0'), 'w2': tensor([[0.1000],
        [0.1000]], device='cuda:0'), 'b2': tensor([[0.]], device='cuda:0')}
tensor([[2.],
        [3.]], device='cuda:0')


In [165]:
y_hat.detach().cpu()

tensor([[0.1000]])

In [166]:
A1

tensor([[0.5000],
        [0.5000]], device='cuda:0')

In [167]:
params['w1']

tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]], device='cuda:0')

In [168]:
params['w1'][0,0]

tensor(0.1000, device='cuda:0')

In [ ]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):
    # Keep this as a scalar tensor to avoid shape mismatch on in-place element updates
    error_signal = (2 * (y - y_hat)).squeeze()

    # Save old w2 values before updating (used for w1/b1 updates)
    w2_00_old = parameters['w2'][0, 0].clone()
    w2_10_old = parameters['w2'][1, 0].clone()

    # Update layer 2
    parameters['w2'][0, 0] += lr * error_signal * A1[0, 0]
    parameters['w2'][1, 0] += lr * error_signal * A1[1, 0]
    parameters['b2'][0, 0] += lr * error_signal

    # Update layer 1
    parameters['w1'][0, 0] += lr * error_signal * w2_00_old * X[0, 0]
    parameters['w1'][0, 1] += lr * error_signal * w2_10_old * X[0, 0]
    parameters['b1'][0, 0] += lr * error_signal * w2_00_old

    parameters['w1'][1, 0] += lr * error_signal * w2_00_old * X[1, 0]
    parameters['w1'][1, 1] += lr * error_signal * w2_10_old * X[1, 0]
    parameters['b1'][1, 0] += lr * error_signal * w2_10_old

    return parameters

In [170]:

#* using loop
params = initialize_parameters([2, 2, 1])
params = {k: v.to(device) for k, v in params.items()}
epochs = 5

for i in range(epochs):
    epoch_loss  = 0
    for j in range(x.shape[0]):
        X_sample = x[j].reshape(2,1).float()
        y_sample = y[j].reshape(1, 1).float()
        y_hat, A1 = L_layer_forward(X_sample, params)
        
        # Loss calculation
        loss = (y_sample - y_hat).pow(2)
        epoch_loss += loss.item()

        params = update_parameters(params, y_sample, y_hat, A1, X_sample)
        
    print(f'Epoch {i+1}/{epochs}, Loss: {epoch_loss/x.shape[0]:.4f}')    



Error signal: tensor(5.8000, device='cuda:0')
Error signal: tensor(7.5287, device='cuda:0')
Error signal: tensor(9.2664, device='cuda:0')
Epoch 1/5, Loss: 14.6824
Error signal: tensor(5.6671, device='cuda:0')
Error signal: tensor(7.2857, device='cuda:0')
Error signal: tensor(8.9247, device='cuda:0')
Epoch 2/5, Loss: 13.7374
Error signal: tensor(5.5049, device='cuda:0')
Error signal: tensor(6.9761, device='cuda:0')
Error signal: tensor(8.4848, device='cuda:0')
Epoch 3/5, Loss: 12.5801
Error signal: tensor(5.3062, device='cuda:0')
Error signal: tensor(6.5835, device='cuda:0')
Error signal: tensor(7.9254, device='cuda:0')
Epoch 4/5, Loss: 11.1926
Error signal: tensor(5.0650, device='cuda:0')
Error signal: tensor(6.0951, device='cuda:0')
Error signal: tensor(7.2318, device='cuda:0')
Epoch 5/5, Loss: 9.5920


### Quick Revision: shape and squeeze()
In PyTorch, **shape** tells how many dimensions a tensor has and how many elements each dimension contains.
Examples:
- `torch.tensor([1,2,3]).shape -> (3,)` (1D tensor)
- `torch.tensor([[1,2,3]]).shape -> (1,3)` (2D tensor)
- `torch.tensor([[[1],[2]]]).shape -> (1,2,1)` (3D tensor)

`squeeze()` removes dimensions whose size is `1`.
- `(1,1)` -> `()` (scalar)
- `(3,1,5)` -> `(3,5)`
- `(2,3)` stays `(2,3)` because there is no dimension of size 1.

Why you used it in your last function:
`error_signal = (2 * (y - y_hat)).squeeze()` changed shape from `(1,1)` to scalar, so updates like `parameters['w2'][0,0] += ...` work without shape mismatch.

In [171]:
# Runnable examples from this notebook context
print('y_sample shape:', y_sample.shape)
print('y_hat shape   :', y_hat.shape)

error_signal_raw = 2 * (y_sample - y_hat)
print('error_signal_raw shape:', error_signal_raw.shape)

error_signal_sq = error_signal_raw.squeeze()
print('after squeeze shape   :', error_signal_sq.shape)
print('value                :', error_signal_sq)

# Extra small examples
a = torch.tensor([[10.0]])          # shape (1,1)
b = a.squeeze()                     # shape ()
c = torch.tensor([[[1.0],[2.0]]])   # shape (1,2,1)
d = c.squeeze()                     # shape (2,)

print('\na shape:', a.shape, '-> b shape:', b.shape)
print('c shape:', c.shape, '-> d shape:', d.shape)

y_sample shape: torch.Size([1, 1])
y_hat shape   : torch.Size([1, 1])
error_signal_raw shape: torch.Size([1, 1])
after squeeze shape   : torch.Size([])
value                : tensor(7.2318, device='cuda:0')

a shape: torch.Size([1, 1]) -> b shape: torch.Size([])
c shape: torch.Size([1, 2, 1]) -> d shape: torch.Size([2])
